1. **Elabore un programa en Python que umbralice una imagen mediante el método general explicado en las clases teóricas. El programa deberá imprimir por consola el valor de umbral obtenido, y finalmente mostrar en pantalla la imagen ya umbralizada.**

**Método general para calcular el umbral:**
1. **Seleccionar un umbral inicial T**
2. **Segmentar la imagen a partir de dicho umbral. G1 es el área con intensidad <T, y G2 el resto**
3. **Calcular la intensidad media de G1 y G2, m1 y m2**
4. **Actualizar el valor de T: T=(1/2)(m1 + m2)**
5. **Repetir los pasos 2 a 4 hasta que el valor de T se estabilice**

![Imagen original](imagenOriginal.jpg)
_Figura 1: Imagen de partida_

![Imagen umbralizada](imagenUmbralizadaGeneral.jpg)
_Figura 2: Imagen umbralizada_

**Código:**

In [21]:
import numpy as np
import cv2, sys

nombreImagen = "p3.png"

#Leemos la imagen y la cargamos en imagenOriginal y hacemos la copia
imagenOriginal = cv2.imread(nombreImagen, cv2.IMREAD_GRAYSCALE)
imagenUmbralizada = cv2.imread(nombreImagen, cv2.IMREAD_GRAYSCALE)

#Si la imagen no se ha podido cargar, terminamos
if (imagenOriginal is None):
    print(" Error al cargar imagen ")
    sys.exit()

#Obtenemos los valores de la imagen en el array dimensiones
dimensiones=imagenOriginal.shape

#Filas o alto
alto = dimensiones[0]

#Columnas o ancho
ancho = dimensiones[1]

#Calculamos las dimensiones de la imagen
dimensiones = ancho * alto

umbral = 127 #valor inicial del umbral

#RESOLVER

def umbralGeneral(alto, ancho, umbral, imagen):
    umbralCalculado = 0

    while(umbral != umbralCalculado):
        if(umbralCalculado != 0):
            umbral = umbralCalculado
        g1 = 0
        g2=0
        m1=0
        m2=0
        for i in range(alto):
            for j in range(ancho):
                if(imagenOriginal[i][j] < umbral):
                    g1 += imagenOriginal[i][j]
                    m1 += 1
                else:
                    g2 += imagenOriginal[i][j]
                    m2 +=1
        umbralCalculado = (g1/m1 + g2/m2)/2
    return umbral


umbral = umbralGeneral(alto,ancho,umbral,imagenOriginal)
    
for i in range(alto):
        for j in range(ancho):
            if(imagenOriginal[i][j] < umbral):
                imagenUmbralizada[i][j] = 0
            else:
                imagenUmbralizada[i][j] = 255
            

#Mostramos las imágenes, guardamos la nueva y mostramos el umbral final
cv2.imshow("Imagen Original", imagenOriginal)
cv2.imshow("Imagen Umbralizada General", imagenUmbralizada)
cv2.imwrite("solucion.png",imagenUmbralizada)
print("El umbral final es", umbral)

#Esperamos a una tecla y cerramos todas las ventanas
cv2.waitKey(0)
cv2.destroyAllWindows()

El umbral final es 132.94177494025314


Para saber cuando ha encontrado el umbral correcto compara el umbral calculado con el último hasta que coincidan sigue calculándolo. La primera comprobación es para cuando se recorre a partir de la segunda vez se vaya asignando el valor del umbralCalculado al umbral para partir desde ese las siguientes veces.
Va recorriendo la imagen alamacenando los calores y por último una vez recorrida la imagen calcula el umbral.

Ese umbral lo asigna abajo como el definitivo y ya comprueba si el valor de la imagen es menor que el umbral lo pone a 0 si no a 255, así con toda la imagen.

EXPLICACIÓN

2. **Escriba un programa en Python que umbralice una imagen mediante el método de Otsu. El funcionamiento del programa ha de ser análogo al del ejercicio 1.**

**Método de umbralización de Otsu**

* **Buscamos el valor k que maximiza la expresión:**

![Otsu 1](otsu-1.jpg)


* **Para una imagen G, con L tonos de gris, y siendo pi los componentes del histograma normalizado, definimos para un umbral k:![Otsu 2](otsu-2.jpg) cumpliendose que: ![Otsu 3](otsu-3.jpg)**

* **Y definimos también:![Otsu 4](otsu-4.jpg)**

![Imagen umbralizada otsu](imagenUmbralizadaOtsu.jpg)
_Figura 3: Imagen umbralizada método de otsu_

**Código:**

In [22]:
import numpy as np
import cv2, sys

nombreImagen = "p3.png"

#Leemos la imagen y la cargamos en imagenOriginal y hacemos la copia
imagenOriginal = cv2.imread(nombreImagen, cv2.IMREAD_GRAYSCALE)
imagenUmbralizada = cv2.imread(nombreImagen, cv2.IMREAD_GRAYSCALE)

#Si la imagen no se ha podido cargar, terminamos
if (imagenOriginal is None):
    print(" Error al cargar imagen ")
    sys.exit()

#Obtenemos los valores de la imagen en el array dimensiones, se podrÃ­a usar img.size (las 2 imagenes son similares)
dimensiones=imagenOriginal.shape

#Filas o alto
alto = dimensiones[0]

#Columnas o ancho
ancho = dimensiones[1]

#De esta manera ya tenemos las dimensiones de la imagen
dimensionesImagen = alto * ancho

#Definimos el array de normalización de valores flotantes
arrayNormalizacion = np.zeros(256, np.float32)

#Array de probabilidad acumulada
pk = np.zeros(256, np.float32)

#Probabilidad acumulada por valor
mk = np.zeros(256, np.float32)

#Sumatoria vector mk
mg = 0

umbral = 0
umbralMax = 0

#RESOLVER
def calculaHistograma(imagen):
    dimensiones = imagen.shape
    alto = dimensiones[0]
    ancho = dimensiones[1]
    vectorHistograma = np.zeros(256, np.uint32)
    for i in range(alto):
        for j in range(ancho):
            vectorHistograma[imagen[i][j]] += 1
    return vectorHistograma

vectorHistograma = calculaHistograma(imagenOriginal)

pk[0] = vectorHistograma[0] / dimensionesImagen
mk[0] = 0

for i in range(1, 256):
    pk[i] = vectorHistograma[i] / dimensionesImagen + pk[i-1]
    mk[i] = i * (vectorHistograma[i] / dimensionesImagen) + mk[i-1]
    mg += i * (vectorHistograma[i] / dimensionesImagen)

for i in range(256):
    if pk[i] == 0 or pk[i] == 1:
        continue 
    expresion = ((mg * pk[i] - mk[i]) ** 2) / (pk[i] * (1 - pk[i]))
    if expresion > umbralMax:
        umbralMax = expresion
        umbral = i

# Aplicamos la umbralización con el umbral calculado
for i in range(alto):
    for j in range(ancho):
        if imagenOriginal[i][j] < umbral:
            imagenUmbralizada[i][j] = 0
        else:
            imagenUmbralizada[i][j] = 255

#Mostramos las imÃ¡genes, guardamos la nueva y mostramos el umbral final
cv2.imshow("Imagen Original", imagenOriginal)
cv2.imshow("Imagen Umbralizada Otsu", imagenUmbralizada)
cv2.imwrite("solucion.png", imagenUmbralizada)
print("El umbral final es", umbral)

#Esperamos a una tecla y cerramos todas las ventanas
cv2.waitKey(0)
cv2.destroyAllWindows()

El umbral final es 133


Lo primero es calcular el histograma de la imagen, una vez lo tengo para calcular pi(k) es ir recorriendo los 256 valores y aplicando el cálculo para los 3 valores, lo hago en un vector para que sea acumulativo y no tenga que repetir cálculos.
Una vez lo tengo hago el cálculo de la fórmula y lo voy almacenando en la variable expresión, quiero ver cuál maximiza la expresión por lo que voy comparándolo y almacenándolo en la variable umbralMax la que vaya cumpliendo que va siendo mayor que la expresión, de esa forma se en que valor tengo el umbral que lo almaceno hacienod lo de umbral = i, para que no se divida por 0 hago la comprobación if pk[i] == 0 or pk[i] == 1:

Una vez calculado el umbral igual que en el método general.

EXPLICACIÓN

3.	**Escriba un programa en Python que particione una imagen en MxN bloques, umbralice cada uno de ellos y grabe en un fichero la imagen resultante umbralizada. Para la resolución de este problema necesitaremos conocer los valores M y N, el método de umbralización (general o de Otsu).**

![division por bloques](bloques.jpg)
_Figura 4: División por bloques_

![Imagen umbralizada por bloques general](imagenUmbralizadaGeneralBloques.jpg)
_Figura 5: Imagen umbralizada por método general usando bloques_

![Imagen umbralizada por bloques otsu](imagenUmbralizadaOtsuBloques.jpg)
_Figura 6: Imagen umbralizada por método Otsu usando bloques_

**Código:**

In [11]:
import numpy as np
import cv2, sys

#Argumentos
nombreImagen = "p3.png"
valorM = 5
valorN = 8
metodo = 1 #0 método general, 1 otsu

#Leemos la imagen y la cargamos en imagenOriginal y hacemos la copia
imagenOriginal = cv2.imread(nombreImagen, cv2.IMREAD_GRAYSCALE)
imagenUmbralizada = cv2.imread(nombreImagen, cv2.IMREAD_GRAYSCALE)

#Si la imagen no se ha podido cargar, terminamos la ejecuciÃ³n
if (imagenOriginal is None):
    print(" Error al cargar imagen ")
    sys.exit()

#Obtenemos los valores de la imagen en el array dimensiones, se podrÃ­a usar img.size (las 2 imagenes son similares)
dimensiones=imagenOriginal.shape

#Filas o alto
alto = dimensiones[0]

#Columnas o ancho
ancho = dimensiones[1]

#De esta manera ya tenemos las dimensiones de la imagen
dimensionesImagen = alto * ancho

#RESOLVER
def umbralGeneral(bloque, umbral):
    umbralCalculado = 0
    while umbral != umbralCalculado:
        if umbralCalculado != 0:
            umbral = umbralCalculado
        g1 = 0
        g2 = 0
        m1 = 0
        m2 = 0
        for i in range(len(bloque)):
            for j in range(len(bloque[0])):
                if bloque[i][j] < umbral:
                    g1 += bloque[i][j]
                    m1 += 1
                else:
                    g2 += bloque[i][j]
                    m2 += 1
        if m1 == 0:
            m1 = 1
        if m2 == 0:
            m2 = 1
            
        umbralCalculado = (g1 / m1 + g2 / m2) / 2
    return umbral

def umbralOtsu(bloque):
    vectorHistograma = np.zeros(256, np.uint32)
    pk = np.zeros(256, np.float64)
    mk = np.zeros(256, np.float64)
    mg = 0.0
    umbralMax = -1
    umbral = 0

    # Calcula el histograma del bloque
    for i in range(len(bloque)):
        for j in range(len(bloque[0])):
            vectorHistograma[bloque[i][j]] += 1

    dimensionesImagen = len(bloque) * len(bloque[0])

    pk[0] = vectorHistograma[0] / dimensionesImagen
    mk[0] = 0

    for i in range(1, 256):
        pk[i] = vectorHistograma[i] / dimensionesImagen + pk[i-1]
        mk[i] = i * (vectorHistograma[i] / dimensionesImagen) + mk[i-1]
        mg += i * (vectorHistograma[i] / dimensionesImagen)

    for i in range(256):
        if pk[i] == 0 or pk[i] == 1:
            continue 
        expresion = ((mg * pk[i] - mk[i]) ** 2) / (pk[i] * (1 - pk[i]))
        if expresion > umbralMax:
            umbralMax = expresion
            umbral = i

    bloque_umbralizado = np.zeros_like(bloque)
    for i in range(len(bloque)):
        for j in range(len(bloque[0])):
            if bloque[i][j] < umbral:
                bloque_umbralizado[i][j] = 0
            else:
                bloque_umbralizado[i][j] = 255

    return bloque_umbralizado

# Calculo las dimensiones de los bloques
bloque_alto = alto // valorM
bloque_ancho = ancho // valorN

# Creo una imagen para almacenar la imagen umbralizada por bloques
imagenUmbralizada = np.zeros_like(imagenOriginal)

# 1
for i in range(valorM):
    for j in range(valorN):
        inicio_y = i * bloque_alto
        fin_y = inicio_y + bloque_alto
        inicio_x = j * bloque_ancho
        fin_x = inicio_x + bloque_ancho

        bloque = imagenOriginal[inicio_y:fin_y, inicio_x:fin_x]

        # Calculamos el umbral usando el método general
        if metodo == 0:
            umbral = umbralGeneral(bloque, 127)

            # Umbralizamos el bloque manualmente
            bloque_umbralizado = np.zeros_like(bloque)
            for x in range(len(bloque)):
                for y in range(len(bloque[0])):
                    if bloque[x][y] >= umbral:
                        bloque_umbralizado[x][y] = 255
        elif metodo == 1:
            bloque_umbralizado = umbralOtsu(bloque)
        
        #2
        imagenUmbralizada[inicio_y:fin_y, inicio_x:fin_x] = bloque_umbralizado

#Mostramos las imagenes, guardamos la nueva y mostramos el umbral final
cv2.imshow("Imagen Original", imagenOriginal)
cv2.imshow("Imagen umbralizada por bloques", imagenUmbralizada)
cv2.imwrite("solucion.png", imagenUmbralizada)

#Esperamos a una tecla y cerramos todas las ventanas
cv2.waitKey(0)
cv2.destroyAllWindows()

Lo explico por partes como están comentadas y numeradas en el código:
Antes defino las dos funciones según el método, ambas le paso el bloque de la imagen que van a procesar y en el caso de la general necesita un umbral inicial, estas dos funciones son como las del anterior ejercicio.
La función del método general devuelve el umbral para posteriormente calcular el bloque asignándole 0 si es menor que el umbral y 255 si es mayor, esto lo hace en el condicional que comprueba si metodo == 0. Para la función del método de Otsu devuelve directamente el bloque umbralizado. 

Por tanto, a partir del comentario 1 lo que hago es extraer el bloque de la imagen calculando con las dimensiones dadas al principio y pasándo como parámetro ese bloque a ambos métodos para calcularlo umbralizado y finalmente en el comentario 2 recomponer la imagen por bloques.


EXPLICACIÓN

4.	**Escriba un programa de umbralización en Python que calcule el umbral para cada píxel a partir de un entorno del mismo de tamaño MxN. Para la resolución de este problema necesitaremos conocer los valores M y N, el método de umbralización (general o de Otsu).**

![entorno](entorno.jpg)
_Figura 7: Entorno de un pixel_

![Imagen umbralizada por entorno general](ImagenUmbralizadaGeneralEntorno.jpg)
_Figura 8: Imagen umbralizada por entorno método general_

![Imagen umbralizada por entorno Otsu](ImagenUmbralizadaOtsuEntorno.jpg)
_Figura 9: Imagen umbralizada por entorno método Otsu_

In [17]:
import numpy as np
import cv2, sys

#Argumentos
nombreImagen = "p3.png"
valorM = 9
valorN = 9
metodo = 0 #0 método general, 1 otsu

#Leemos la imagen y la cargamos en imagenOriginal y hacemos la copia
imagenOriginal = cv2.imread(nombreImagen, cv2.IMREAD_GRAYSCALE)
imagenUmbralizada = cv2.imread(nombreImagen, cv2.IMREAD_GRAYSCALE)

#Si la imagen no se ha podido cargar, terminamos
if (imagenOriginal is None):
    print(" Error al cargar imagen ")
    sys.exit()

#Obtenemos los valores de la imagen en el array dimensiones, se podrÃ­a usar img.size (las 2 imagenes son similares)
dimensiones=imagenOriginal.shape

#Filas o alto
alto = dimensiones[0]

#Columnas o ancho
ancho = dimensiones[1]

#De esta manera ya tenemos las dimensiones de la imagen
dimensionesImagen = alto * ancho

#RESOLVER
def umbralPixelGeneral(pixel, umbral):
    if pixel < umbral:
        return 0
    else:
        return 255

# Función para calcular el umbral de un pixel usando el método de Otsu
def umbralPixelOtsu(pixel, bloque):
    vectorHistograma = np.zeros(256, np.uint32)
    pk = np.zeros(256, np.float64)
    mk = np.zeros(256, np.float64)
    mg = 0.0
    umbralMax = -1
    umbral = 0

    # Calculo el histograma del bloque
    for i in range(len(bloque)):
        for j in range(len(bloque[0])):
            vectorHistograma[bloque[i][j]] += 1

    dimensionesImagen = len(bloque) * len(bloque[0])

    pk[0] = vectorHistograma[0] / dimensionesImagen
    mk[0] = 0

    # Calculo pk y mk para todos los niveles de gris
    for i in range(1, 256):
        pk[i] = vectorHistograma[i] / dimensionesImagen + pk[i-1]
        mk[i] = i * (vectorHistograma[i] / dimensionesImagen) + mk[i-1]
        mg += i * (vectorHistograma[i] / dimensionesImagen)

    # Encuentro el umbral según el método de Otsu
    for i in range(256):
        if pk[i] == 0 or pk[i] == 1:
            continue 
        expresion = ((mg * pk[i] - mk[i]) ** 2) / (pk[i] * (1 - pk[i]))
        if expresion > umbralMax:
            umbralMax = expresion
            umbral = i

    # Aplico la umbralización con el umbral calculado
    if pixel < umbral:
        return 0
    else:
        return 255

# Itero sobre los píxeles de la imagen
for i in range(alto):
    for j in range(ancho):
        # Definino las coordenadas del entorno del pixel actual
        inicio_y = max(0, i - (valorM // 2))
        fin_y = min(alto, i + (valorM // 2) + 1)
        inicio_x = max(0, j - (valorN // 2))
        fin_x = min(ancho, j + (valorN // 2) + 1)

        # Extraigo el entorno del pixel
        entorno = imagenOriginal[inicio_y:fin_y, inicio_x:fin_x]

        # Calculo el umbral para el pixel
        if metodo == 0:
            imagenUmbralizada[i][j] = umbralPixelGeneral(imagenOriginal[i][j], umbral)
        elif metodo == 1:
            imagenUmbralizada[i][j] = umbralPixelOtsu(imagenOriginal[i][j], entorno)
            
#Mostramos las imágenes, guardamos la nueva
cv2.imshow("Imagen Original", imagenOriginal)
cv2.imshow("Imagen umbralizada por entorno", imagenUmbralizada)
cv2.imwrite("solucion.png", imagenUmbralizada)

#Esperamos a una tecla y cerramos todas las ventanas
cv2.waitKey(0)
cv2.destroyAllWindows()


Lo explico en los comentarios.

EXPLICACIÓN